<a href="https://colab.research.google.com/github/ITDeema/DataScience/blob/main/DATASCIENCEPHASE2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Phase 1**

In [1]:
!apt-get update -y
!apt-get install -y chromium-chromedriver
!pip install -q selenium pandas openpyxl
!pip install google-search-results


Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 https://cli.github.com/packages stable/main amd64 Packages [356 B]
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:8 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [89.0 kB]
Get:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [10.1 MB]
Get:11 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,292 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/multiverse amd64 Packages [61.6 kB]
Get:13 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,864 kB]
G

In [2]:
from serpapi import GoogleSearch
import pandas as pd
import time

api_key = "783dcf17ea65c6a84a44cceb4fd71c5129f14f0eefc1d4cfc5c3249f8a095bd4"

queries = [
    # Makkah
    "Hotels in Makkah",
    "Hotels near Masjid al Haram",
    "Hotels in Ajyad Makkah",
    "Hotels in Aziziyah Makkah",
    "Hotels in Al Misfalah Makkah",
    "Hotels in Al Shubaikah Makkah",
    "Hotels in Al Naseem Makkah",
    "Hotels in Kudai Makkah",
    "Hotels in Jarwal Makkah",

    # Madinah
    "Hotels in Madinah",
    "Hotels near Al Masjid an Nabawi",
    "Hotels in Central Madinah",
    "Hotels in Al Haram Madinah",
    "Hotels in Quba Madinah",
    "Hotels in Al Azhari Madinah",
    "Hotels in Al Aridh Madinah",
    "Hotels in Bani Khidrah Madinah"
]

date_ranges = [
    ("2026-02-15", "2026-02-20"),
    ("2026-03-10", "2026-03-15"),
    ("2026-03-20", "2026-03-25"),
    ("2026-05-20", "2026-05-25"),
    ("2026-06-10", "2026-06-15"),
    ("2026-07-01", "2026-07-06")
]

all_hotels = []

for checkin, checkout in date_ranges:
    for q in queries:
        for start in [0, 20, 40, 60, 80]:

            params = {
                "engine": "google_hotels",
                "q": q,
                "check_in_date": checkin,
                "check_out_date": checkout,
                "adults": 2,
                "currency": "SAR",
                "gl": "sa",
                "hl": "ar",
                "start": start,
                "api_key": api_key
            }

            search = GoogleSearch(params)
            results = search.get_dict()

            for h in results.get("properties", []):


                if checkin.startswith("2026-03"):
                    season_type = "Ramadan Season"
                    context_text = "High spiritual tourism season during Ramadan"
                elif checkin.startswith("2026-06"):
                    season_type = "Hajj Peak"
                    context_text = "Peak pilgrimage demand period"
                elif checkin.startswith("2026-05"):
                    season_type = "Pre-Hajj"
                    context_text = "Pre-Hajj booking increase period"
                else:
                    season_type = "Regular Umrah"
                    context_text = "Regular travel season with moderate hotel demand"

                address = h.get("address")
                if not address:
                    gps = h.get("gps_coordinates", {})
                    lat = gps.get("latitude")
                    lng = gps.get("longitude")
                    if lat and lng:
                        address = f"Lat: {lat}, Lng: {lng}"
                    else:
                        address = None


                city = "Madinah" if "Madinah" in q else "Makkah"

                all_hotels.append({
                    "Hotel Name": h.get("name"),
                    "Price (SAR)": h.get("rate_per_night", {}).get("lowest"),
                    "Rating": h.get("overall_rating"),
                    "Reviews": h.get("reviews"),
                    "Address": address,
                    "City": city,
                    "Check-in": checkin,
                    "Check-out": checkout,
                    "Search Area": q,
                    "Season_Type": season_type,
                    "Religious_Context": context_text
                })

            time.sleep(1)

df_final = pd.DataFrame(all_hotels)

df_final = df_final.drop_duplicates(
    subset=["Hotel Name", "Address", "Check-in"]
).reset_index(drop=True)

print("Total rows collected:", len(df_final))

df_final.to_csv("makkah_madinah_hajj_umrah_2026_with_unstructured.csv", index=False)

df_final.head()


Total rows collected: 675


,Hotel Name,Price (SAR),Rating,Reviews,Address,City,Check-in,Check-out,Search Area,Season_Type,Religious_Context
0,فندق بولمان زمزم مكة,None,4.6,61117.0,"Lat: 21.4196355, Lng: 39.8237376",Makkah,2026-05-20,2026-05-25,Hotels in Makkah,Pre-Hajj,Pre-Hajj booking increase period
1,فندق وأبراج مكة,None,4.8,62376.0,"Lat: 21.419156100000002, Lng: 39.8236886",Makkah,2026-05-20,2026-05-25,Hotels in Makkah,Pre-Hajj,Pre-Hajj booking increase period
2,فندق دار التوحيد إنتركونتيننتال,None,4.8,35868.0,"Lat: 21.4210048, Lng: 39.822747299999996",Makkah,2026-05-20,2026-05-25,Hotels in Makkah,Pre-Hajj,Pre-Hajj booking increase period
3,فندق ثري بيرلز مصلي,‏١٬٣٣٤ ر.س.‏,4.2,1258.0,"Lat: 21.423771000000002, Lng: 39.809525199999996",Makkah,2026-05-20,2026-05-25,Hotels in Makkah,Pre-Hajj,Pre-Hajj booking increase period
4,دبل ترى باي هيلتون جبل عمر مكة,None,4.5,8942.0,"Lat: 21.4231692, Lng: 39.8178141",Makkah,2026-05-20,2026-05-25,Hotels in Makkah,Pre-Hajj,Pre-Hajj booking increase period


In [3]:
from serpapi import GoogleSearch
import pandas as pd
import time

api_key = "783dcf17ea65c6a84a44cceb4fd71c5129f14f0eefc1d4cfc5c3249f8a095bd4"

queries = [
    "Hotels in Taif",
    "Hotels in Taif for Hajj visitors",
    "Hotels in Taif near Miqat",
    "Umrah visitor hotels in Taif",
    "Budget hotels in Taif for pilgrims",
    "Luxury hotels in Taif during Hajj season",
    "Hotels in Taif near Al Sail Al Kabir Miqat"
]

date_ranges = [
    ("2026-02-15", "2026-02-20"),
    ("2026-03-10", "2026-03-15"),
    ("2026-03-20", "2026-03-25"),
    ("2026-05-20", "2026-05-25"),
    ("2026-06-10", "2026-06-15"),
    ("2026-07-01", "2026-07-06")
]

all_hotels = []

for checkin, checkout in date_ranges:
    for q in queries:
        for start in [0, 20, 40]:

            params = {
                "engine": "google_hotels",
                "q": q,
                "check_in_date": checkin,
                "check_out_date": checkout,
                "adults": 2,
                "currency": "SAR",
                "gl": "sa",
                "hl": "ar",
                "start": start,
                "api_key": api_key
            }

            search = GoogleSearch(params)
            results = search.get_dict()

            for h in results.get("properties", []):


                if checkin.startswith("2026-03"):
                    season_type = "Ramadan Season"
                    context_text = "Ramadan spiritual tourism season affecting Taif hotels"
                elif checkin.startswith("2026-06"):
                    season_type = "Hajj Peak"
                    context_text = "Hajj season overflow demand with pilgrims staying in Taif"
                elif checkin.startswith("2026-05"):
                    season_type = "Pre-Hajj"
                    context_text = "Pre-Hajj increasing accommodation demand in Taif"
                else:
                    season_type = "Regular Umrah"
                    context_text = "Regular Umrah related travel near Makkah region"

                address = h.get("address")
                if not address:
                    gps = h.get("gps_coordinates", {})
                    lat = gps.get("latitude")
                    lng = gps.get("longitude")
                    if lat and lng:
                        address = f"Lat: {lat}, Lng: {lng}"
                    else:
                        address = None

                all_hotels.append({
                    "Hotel Name": h.get("name"),
                    "City": "Taif",
                    "Price (SAR)": h.get("rate_per_night", {}).get("lowest"),
                    "Rating": h.get("overall_rating"),
                    "Reviews": h.get("reviews"),
                    "Address": address,
                    "Check-in": checkin,
                    "Check-out": checkout,
                    "Search Area": q,
                    "Season_Type": season_type,
                    "Religious_Context": context_text
                })

            time.sleep(1)

df_taif = pd.DataFrame(all_hotels)

df_taif = df_taif.drop_duplicates(
    subset=["Hotel Name", "Address", "Check-in"]
).reset_index(drop=True)

print("Total Taif rows:", len(df_taif))

df_taif.to_csv("taif_hajj_umrah_2026.csv", index=False)

df_taif.head()

Total Taif rows: 193


,Hotel Name,City,Price (SAR),Rating,Reviews,Address,Check-in,Check-out,Search Area,Season_Type,Religious_Context
0,Al Dana Plaza Villas - Three-Bedroom Villa wit...,Taif,‏٣٦٨ ر.س.‏,NaN,NaN,"Lat: 21.223649978637695, Lng: 40.41001892089844",2026-05-20,2026-05-25,Hotels in Taif,Pre-Hajj,Pre-Hajj increasing accommodation demand in Taif
1,فندق أوالف - 4 نجوم,Taif,‏٤١٥ ر.س.‏,4.1,3101.0,"Lat: 21.2882359, Lng: 40.419680299999996",2026-05-20,2026-05-25,Hotels in Taif,Pre-Hajj,Pre-Hajj increasing accommodation demand in Taif
2,منتجع أرائك للفلل الفندقية الفاخرة,Taif,‏٤٣٢ ر.س.‏,4.2,2767.0,"Lat: 21.2114062, Lng: 40.3967074",2026-05-20,2026-05-25,Hotels in Taif,Pre-Hajj,Pre-Hajj increasing accommodation demand in Taif
3,برود,Taif,‏٣٧٩ ر.س.‏,4.2,1215.0,"Lat: 21.2826265, Lng: 40.41373670000001",2026-05-20,2026-05-25,Hotels in Taif,Pre-Hajj,Pre-Hajj increasing accommodation demand in Taif
4,الديوان النجدي للشقق المخدومة اقتصادي - One-Be...,Taif,‏٢٩٦ ر.س.‏,NaN,NaN,"Lat: 21.484140396118164, Lng: 40.48775863647461",2026-05-20,2026-05-25,Hotels in Taif,Pre-Hajj,Pre-Hajj increasing accommodation demand in Taif


In [4]:
import pandas as pd

df_main = pd.read_csv("makkah_madinah_hajj_umrah_2026_with_unstructured.csv")
df_taif = pd.read_csv("taif_hajj_umrah_2026.csv")

print("Rows before merge:")
print("Main file:", len(df_main))
print("Taif file:", len(df_taif))

df_main.columns = df_main.columns.str.strip()
df_taif.columns = df_taif.columns.str.strip()

df_combined = pd.concat([df_main, df_taif], ignore_index=True, sort=False)

df_combined = df_combined.drop_duplicates(
    subset=["Hotel Name", "Address", "Check-in"]
).reset_index(drop=True)

print("Final merged rows:", len(df_combined))

df_combined.to_csv("FINAL_HAJJ_UMRAH_ALL_CITIES_2026.csv", index=False)

df_combined.head()


Rows before merge:
Main file: 675
Taif file: 193
Final merged rows: 868


,Hotel Name,Price (SAR),Rating,Reviews,Address,City,Check-in,Check-out,Search Area,Season_Type,Religious_Context
0,فندق بولمان زمزم مكة,NaN,4.6,61117.0,"Lat: 21.4196355, Lng: 39.8237376",Makkah,2026-05-20,2026-05-25,Hotels in Makkah,Pre-Hajj,Pre-Hajj booking increase period
1,فندق وأبراج مكة,NaN,4.8,62376.0,"Lat: 21.419156100000002, Lng: 39.8236886",Makkah,2026-05-20,2026-05-25,Hotels in Makkah,Pre-Hajj,Pre-Hajj booking increase period
2,فندق دار التوحيد إنتركونتيننتال,NaN,4.8,35868.0,"Lat: 21.4210048, Lng: 39.822747299999996",Makkah,2026-05-20,2026-05-25,Hotels in Makkah,Pre-Hajj,Pre-Hajj booking increase period
3,فندق ثري بيرلز مصلي,‏١٬٣٣٤ ر.س.‏,4.2,1258.0,"Lat: 21.423771000000002, Lng: 39.809525199999996",Makkah,2026-05-20,2026-05-25,Hotels in Makkah,Pre-Hajj,Pre-Hajj booking increase period
4,دبل ترى باي هيلتون جبل عمر مكة,NaN,4.5,8942.0,"Lat: 21.4231692, Lng: 39.8178141",Makkah,2026-05-20,2026-05-25,Hotels in Makkah,Pre-Hajj,Pre-Hajj booking increase period


# **Phase 2**

In [5]:
import numpy as np
#create a copy of the data
df_raw = pd.read_csv("Unstructured _Data.csv")

df = df_raw.copy()

FileNotFoundError: [Errno 2] No such file or directory: 'Unstructured _Data.csv'

In [ ]:
df.head()
df.info()
df.describe()

# **Data Cleaning**

In [ ]:
# Calculate missing values percentage
missing_table = pd.DataFrame({
    "Missing Count": df.isnull().sum(),
    "Missing %": df.isnull().mean() * 100
})

missing_table.sort_values("Missing %", ascending=False)

In [ ]:
# Remove duplicates based on specific booking details (Address, Check-in, Check-out) rather than just the hotel name
duplicate_count = df.duplicated(subset=['Address', 'Check-in', 'Check-out']).sum()
print(f"Number of duplicates found: {duplicate_count}")

df.drop_duplicates(subset=['Address', 'Check-in', 'Check-out'], inplace=True)
df.drop_duplicates(inplace=True)


print(df.shape)

In [ ]:
import re
# Clean the Price column: convert Arabic numerals to English and extract only digits

def arabic_to_english_number(text):
    arabic_numbers = "٠١٢٣٤٥٦٧٨٩"
    english_numbers = "0123456789"
    translator = str.maketrans(arabic_numbers, english_numbers)
    return text.translate(translator)

def clean_price(value):
    if pd.isna(value):
        return None
    value = str(value)
    value = arabic_to_english_number(value)
    value = re.sub(r"[^\d]", "", value)
    if value == "":
        return None
    return float(value)

df["Price (SAR)"] = df["Price (SAR)"].apply(clean_price)

print(df["Price (SAR)"])

In [ ]:
# Impute missing values:
# - Median for Price
# - Mean for Rating
# - 0 for Reviews

df["Price (SAR)"] = df["Price (SAR)"].fillna(df["Price (SAR)"].median())
df["Rating"] = df["Rating"].fillna(df["Rating"].mean()).round(1)
df["Reviews"] = df["Reviews"].fillna(0)

print(df.isnull().sum())

In [ ]:
# Extract Latitude and Longitude from the Address string

df[["Latitude", "Longitude"]] = df["Address"].str.extract(r"Lat:\s*(-?[\d\.]+),\s*Lng:\s*(-?[\d\.]+)")

df["Latitude"] = pd.to_numeric(df["Latitude"])
df["Longitude"] = pd.to_numeric(df["Longitude"])

print(df[["Address", "Latitude", "Longitude"]].head())

In [ ]:
# Standardize date columns to datetime objects
df["Check-in"] = pd.to_datetime(df["Check-in"])
df["Check-out"] = pd.to_datetime(df["Check-out"])
print(df[["Check-in", "Check-out"]])

In [ ]:
df.info()
df.describe()
df.isnull().sum()

In [ ]:
df.to_csv("CLEANED_HAJJ_UMRAH.csv", index=False)

# **Data Preprocessing**

In [ ]:
# Create duration of stay
df["Stay_Duration"] = (df["Check-out"] - df["Check-in"]).dt.days

# Create a Price category
df["Price_Category"] = pd.cut(
    df["Price (SAR)"],
    bins=[0, 300, 800, 1500, 10000],
    labels=["Budget", "Mid-Range", "Premium", "Luxury"]
)

In [ ]:
# One-Hot Encoding
df_encoded = pd.get_dummies(
    df,
    columns=["City", "Season_Type", "Price_Category"],
    drop_first=False
)

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

numeric_features = ["Price (SAR)", "Rating", "Reviews", "Stay_Duration"]

df_encoded[numeric_features] = scaler.fit_transform(df_encoded[numeric_features])

In [ ]:
df_encoded.to_csv("PROCESSED_HAJJ_UMRAH_2026.csv", index=False)

# **EDA**

In [ ]:
import numpy as np
import pandas as pd
#create a copy of the data
df_raw = pd.read_csv("PROCESSED_HAJJ_UMRAH_2026.csv")

df = df_raw.copy()


In [ ]:
df["Price (SAR)"] = pd.to_numeric(df["Price (SAR)"], errors="coerce")
df["Rating"] = pd.to_numeric(df["Rating"], errors="coerce")


In [ ]:
df[["Price (SAR)", "Rating"]].describe()

Generated descriptive statistics for Price and Rating.
The mean is approximately 0 and the standard deviation is close to 1,
 confirming successful standardization.
 The maximum price value is significantly higher than the rest,
indicating the presence of an extreme price outlier.

In [ ]:
df[["City_Makkah","City_Madinah","City_Taif"]].sum()


Makkah contains the highest number of hotels, followed by Madinah, while Taif has the lowest number.
This suggests that hotel supply is concentrated in Makkah, likely due to higher demand.

In [ ]:
import matplotlib.pyplot as plt
plt.figure()
city_counts = df[["City_Makkah","City_Madinah","City_Taif"]].sum()
plt.bar(["Makkah","Madinah","Taif"], city_counts)
plt.title("Number of Hotels by City")
plt.show()


The visualization clearly shows that Makkah dominates in hotel availability,
indicating it is the primary destination among the three cities.

In [ ]:
df[[
    "Season_Type_Hajj Peak",
    "Season_Type_Pre-Hajj",
    "Season_Type_Ramadan Season",
    "Season_Type_Regular Umrah"
]].sum()


Ramadan season has the highest number of hotels, while other seasons are relatively similar in distribution.
This suggests that Ramadan is a peak operational period.

In [ ]:
plt.figure()
season_counts = df[
    ["Season_Type_Hajj Peak",
     "Season_Type_Pre-Hajj",
     "Season_Type_Ramadan Season",
     "Season_Type_Regular Umrah"]
].sum()

plt.bar(["Hajj Peak","Pre-Hajj","Ramadan","Regular Umrah"], season_counts)
plt.xticks(rotation=45)
plt.title("Number of Hotels by Season")
plt.show()


The bar chart confirms that hotel availability increases during Ramadan,
indicating higher demand compared to other seasons.

In [ ]:
df[["Price (SAR)", "Rating"]].corr()


There is almost no correlation between price and rating,
indicating that higher-priced hotels do not necessarily receive better ratings.

In [ ]:
plt.figure()
plt.scatter(df["Price (SAR)"], df["Rating"])
plt.title("Price vs Rating")
plt.xlabel("Standardized Price")
plt.ylabel("Standardized Rating")
plt.axhline(0)
plt.axvline(0)
plt.show()


The scatter plot shows no clear linear relationship between price and rating.
Most data points cluster around average values, with a noticeable outlier at a very high price.

In [ ]:
summary = pd.DataFrame({
    "City": ["Makkah", "Madinah", "Taif"],
    "Avg_Price": [
        df[df["City_Makkah"] == 1]["Price (SAR)"].mean(),
        df[df["City_Madinah"] == 1]["Price (SAR)"].mean(),
        df[df["City_Taif"] == 1]["Price (SAR)"].mean()
    ],
    "Avg_Rating": [
        df[df["City_Makkah"] == 1]["Rating"].mean(),
        df[df["City_Madinah"] == 1]["Rating"].mean(),
        df[df["City_Taif"] == 1]["Rating"].mean()
    ]
})

In [ ]:
import matplotlib.pyplot as plt

plt.figure()
plt.bar(summary["City"], summary["Avg_Price"])
plt.title("Average Standardized Price by City")
plt.xlabel("City")
plt.ylabel("Standardized Price")
plt.axhline(0)
plt.show()


Makkah has the highest average standardized price, while Taif has the lowest.
This indicates that hotels in Makkah tend to be more expensive on average.

In [ ]:
plt.figure()
plt.bar(summary["City"], summary["Avg_Rating"])
plt.title("Average Standardized Rating by City")
plt.xlabel("City")
plt.ylabel("Standardized Rating")
plt.axhline(0)
plt.show()


Madinah has the highest average rating
followed by Taif, while Makkah has the lowest average rating among the three cities.

In [ ]:
season_price = pd.DataFrame({
    "Season": ["Hajj Peak", "Pre-Hajj", "Ramadan", "Regular Umrah"],
    "Avg_Price": [
        df[df["Season_Type_Hajj Peak"] == 1]["Price (SAR)"].mean(),
        df[df["Season_Type_Pre-Hajj"] == 1]["Price (SAR)"].mean(),
        df[df["Season_Type_Ramadan Season"] == 1]["Price (SAR)"].mean(),
        df[df["Season_Type_Regular Umrah"] == 1]["Price (SAR)"].mean()
    ]
})

plt.figure()
plt.bar(season_price["Season"], season_price["Avg_Price"])
plt.title("Average Standardized Price by Season")
plt.xticks(rotation=45)
plt.axhline(0)
plt.show()


The bar chart shows the average standardized hotel price across different seasons.
Pre-Hajj has the highest average price, followed by Ramadan.
Hajj Peak and Regular Umrah show lower average prices.
This indicates that price increases tend to occur before the peak pilgrimage period.

In [ ]:
plt.figure(figsize=(8,6))
plt.scatter(df["Longitude"], df["Latitude"], c=df["Price (SAR)"], cmap="viridis", alpha=0.6)
plt.colorbar(label="Price (SAR)")
plt.title("Hotel Price Distribution by Location")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.show()

The scatter plot shows the geographical distribution of hotels using their GPS coordinates (latitude and longitude). Hotels appear grouped into clusters that correspond to different city locations. The color scale represents hotel prices, indicating that price levels may vary depending on geographical location, particularly in areas associated with major religious destinations.

In [ ]:
plt.figure(figsize=(8,6))
plt.scatter(df["Latitude"], df["Price (SAR)"], alpha=0.6)
plt.title("Relationship Between Location and Price")
plt.xlabel("Latitude")
plt.ylabel("Price (SAR)")
plt.show()

The scatter plot illustrates the relationship between geographical location (latitude) and hotel price. The distribution of points shows that hotel prices vary across different geographical areas. This suggests that location may play a role in hotel pricing, especially in regions closer to important religious sites.

# Secondary EDA

In [ ]:
import pandas as pd

secondary_df = pd.read_csv("hotel_bookings.csv")

secondary_df.head()

In [ ]:
secondary_df.info();
secondary_df.describe();


The secondary dataset contains hotel booking records from an external source dataset.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))
sns.histplot(secondary_df['adr'], bins=30)
plt.title("Distribution of hotel prices ")
plt.xlabel("Avarge Daily Rate")
plt.show()

This histogram shows the distribution of hotel prices in the secondary dataset.


In [ ]:
sns.countplot(x='hotel', data=secondary_df)
plt.title("Distribution of Hotel Types")
plt.show()

This chart shows the distribution of hotel types in the dataset, including Resort Hotels and City Hotels.

In [ ]:
sns.scatterplot(x='adults', y='adr', data=secondary_df)
plt.title("Relationship Between Adults and Hotel Price")
plt.show()

The scatter plot explores whether the number of adults influences hotel prices.

In [ ]:
plt.figure(figsize=(8,6))
sns.heatmap(secondary_df[['adr','adults','lead_time','stays_in_week_nights']].corr(), annot=True)
plt.title("Correlation Between Booking Variables")
plt.show()

The correlation matrix highlights relationships between booking features and hotel pricing.

The secondary dataset comes from an external hotel booking dataset used in research.
It includes information about hotel type, price (ADR), and customer booking details.
This data helps provide additional insight into hotel pricing patterns.

Limitations:

The dataset is not specific to religious tourism.

Data may come from different countries and time periods.

Booking patterns may differ from Hajj and Umrah travel behavior.

## Compare Results of Datasets

firts, we run descriptive statistics for numerical variables in the secondary dataset, such as lead time, number of adults, waiting days, and ADR (Average Daily Rate).



In [ ]:
secondary_df.describe()

then we compare the average hotel price in the primary dataset with the average daily rate (ADR) from the secondary dataset.


In [ ]:
primary_df=pd.read_csv("CLEANED_HAJJ_UMRAH.csv")
primary_df["Price (SAR)"] = pd.to_numeric(primary_df["Price (SAR)"], errors="coerce")

In [ ]:
import pandas as pd

price_comparison = pd.DataFrame({
    "Dataset": ["Primary Dataset", "Secondary Dataset"],
    "Average Price": [
       primary_df["Price (SAR)"].mean(),
        secondary_df["adr"].mean()
    ]
})

print(price_comparison)

### Average price comparison chart

In [ ]:
import matplotlib.pyplot as plt

primary_price = primary_df["Price (SAR)"].mean()
secondary_price = secondary_df["adr"].mean()

datasets = ["Primary Dataset", "Secondary Dataset"]
prices = [primary_price, secondary_price]

plt.figure(figsize=(6,4))

plt.bar(datasets, prices)

plt.title("Average Price Comparison Between Datasets")
plt.ylabel("Average Price (SAR)")

plt.show()

**Primary Dataset Price Distribution**

In [ ]:
plt.figure(figsize=(6,4))

primary_prices = primary_df["Price (SAR)"]
primary_prices = primary_prices[primary_prices < 5000]  # لقيم المتطرفة

plt.hist(primary_prices, bins=30)

plt.title("Primary Dataset Price Distribution")
plt.xlabel("Price (SAR)")
plt.ylabel("Frequency")

plt.show()

**Secondary Dataset ADR Distribution**

In [ ]:
plt.figure(figsize=(6,4))

secondary_prices = secondary_df["adr"]
secondary_prices = secondary_prices[secondary_prices < 500]
plt.hist(secondary_prices, bins=30)

plt.title("Secondary Dataset ADR Distribution")
plt.xlabel("ADR")
plt.ylabel("Frequency")

plt.show()

#Primary Data Insights


Rating Distribution

In [ ]:
plt.figure(figsize=(6,4))

plt.hist(primary_df["Rating"], bins=20)

plt.title("Rating Distribution in Primary Dataset")
plt.xlabel("Rating")
plt.ylabel("Frequency")

plt.show()

Reviews distribution


In [ ]:
plt.figure(figsize=(6,4))

plt.hist(primary_df["Reviews"], bins=20)

plt.title("Reviews Distribution")
plt.xlabel("Number of Reviews")
plt.ylabel("Frequency")

plt.show()

# Secondary Data Insights


Lead Time Distribution:
its how many days in advance a booking was made before the arrival date.


In [ ]:
plt.figure(figsize=(6,4))

plt.hist(secondary_df["lead_time"], bins=30)

plt.title("Lead Time Distribution")
plt.xlabel("Lead Time (days)")
plt.ylabel("Frequency")

plt.show()

Cancellation rate


In [ ]:
cancel_rate = secondary_df["is_canceled"].mean()

print("Cancellation Rate:", cancel_rate)

# **Phase 3**

**Baseline Model (Linear Regression):**

In [11]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression


df = pd.read_csv("PROCESSED_HAJJ_UMRAH_2026.csv")

# Convert dates and calculate the stay duration (Stay_Duration)
df["Stay_Duration"] = (pd.to_datetime(df["Check-out"]) - pd.to_datetime(df["Check-in"])).dt.days

# Define the independent variables (features) and the target variable
features = ['Rating', 'Reviews', 'Stay_Duration',
            'City_Makkah', 'City_Madinah', 'City_Taif',
            'Season_Type_Hajj Peak', 'Season_Type_Pre-Hajj',
            'Season_Type_Ramadan Season', 'Season_Type_Regular Umrah']

X = df[features]
y = df['Price (SAR)']



# Split the data into training and testing sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


# Define and train the Baseline Model (Linear Regression)
baseline_model = LinearRegression()
baseline_model.fit(X_train, y_train)

# Predict the prices for the test set (Your task ends here)
y_pred = baseline_model.predict(X_test)

FileNotFoundError: [Errno 2] No such file or directory: 'PROCESSED_HAJJ_UMRAH_2026.csv'

**Advanced Model : Decision Tree Regressor**

In [ ]:

from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np
import matplotlib.pyplot as plt

#train decision tree model
dt_model = DecisionTreeRegressor(random_state=42)
dt_model.fit(X_train, y_train)

# make prediction using the model
y_pred_dt = dt_model.predict(X_test)

# Evaluate model performance (RMSE)
mae_dt = mean_absolute_error(y_test, y_pred_dt)
rmse_dt = np.sqrt(mean_squared_error(y_test, y_pred_dt))

print("Decision Tree MAE:", mae_dt)
print("Decision Tree RMSE:", rmse_dt)

# Graph
plt.figure()
plt.scatter(y_test, y_pred_dt)
plt.xlabel("Actual Prices")
plt.ylabel("Predicted Prices")
plt.title("Decision Tree Predictions")
plt.show()

In [ ]:
baseline_pred = [y_train.mean()] * len(y_test)

from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np
import matplotlib.pyplot as plt

baseline_mae = mean_absolute_error(y_test, baseline_pred)
baseline_rmse = np.sqrt(mean_squared_error(y_test, baseline_pred))
#compare with baseline model
print("Baseline MAE:", baseline_mae)
print("Baseline RMSE:", baseline_rmse)


mae_dt = mean_absolute_error(y_test, y_pred_dt)
rmse_dt = np.sqrt(mean_squared_error(y_test, y_pred_dt))

print("Decision Tree MAE:", mae_dt)
print("Decision Tree RMSE:", rmse_dt)


models = ["Baseline", "Decision Tree"]
rmse_values = [baseline_rmse, rmse_dt]

plt.figure()
plt.bar(models, rmse_values)
plt.title("Baseline vs Decision Tree (RMSE)")
plt.xlabel("Models")
plt.ylabel("RMSE")
plt.show()

 The Decision Tree performed worse than the baseline model, as it produced a higher RMSE, indicating less accurate predictions

**Advanced** **Model** : **Random** **Forest** **Regressor**

In [ ]:

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt


df = pd.read_csv("PROCESSED_HAJJ_UMRAH_2026.csv")


features = [
    'Rating', 'Reviews', 'Stay_Duration',
    'Latitude', 'Longitude',
    'City_Makkah', 'City_Madinah', 'City_Taif',
    'Season_Type_Hajj Peak', 'Season_Type_Pre-Hajj',
    'Season_Type_Ramadan Season', 'Season_Type_Regular Umrah'
]

X = df[features]
y = df['Price (SAR)']


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


model_rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=10,
    random_state=42
)


model_rf.fit(X_train, y_train)


y_pred_rf = model_rf.predict(X_test)


mse_rf = mean_squared_error(y_test, y_pred_rf)
r2_rf = r2_score(y_test, y_pred_rf)

print("Random Forest Results:")
print("MSE:", mse_rf)
print("R2 Score:", r2_rf)


plt.figure(figsize=(6,6))
plt.scatter(y_test, y_pred_rf, alpha=0.5)
plt.xlabel("Actual Prices")
plt.ylabel("Predicted Prices")
plt.title("Random Forest: Actual vs Predicted Prices")
plt.show()


importance = model_rf.feature_importances_
features_names = X.columns

feat_imp = pd.Series(importance, index=features_names).sort_values(ascending=False)

feat_imp.head(10).plot(kind='bar')
plt.title("Top Important Features (Random Forest)")
plt.show()

A Random Forest Regressor was trained to predict hotel prices, but it performed worse than the baseline model (Linear Regression), achieving a negative R² (-0.28), indicating poor predictive accuracy.

##  Model Evaluation

### Purpose
In this section, the performance of all developed models is evaluated to identify the best-performing model based on prediction accuracy and error values.

---

###  Evaluation Metrics

- **Mean Absolute Error (MAE)**  
- **Mean Squared Error (MSE)**  
- **Root Mean Squared Error (RMSE)**  
- **R² Score**

---

###  Why These Metrics?
These metrics help in understanding how well each model predicts hotel prices and how much error is present in the predictions.

In [9]:
# ==========================================
# Final Model Evaluation (Student 4 Task)
# ==========================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Function to evaluate regression models
def evaluate_model(model_name, y_true, y_predicted):
    mae = mean_absolute_error(y_true, y_predicted)
    mse = mean_squared_error(y_true, y_predicted)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_predicted)

    return {
        "Model": model_name,
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "R² Score": r2
    }

# Mean Baseline prediction
baseline_pred = [y_train.mean()] * len(y_test)

# Evaluate all models
results = [
    evaluate_model("Mean Baseline", y_test, baseline_pred),
    evaluate_model("Linear Regression", y_test, y_pred),
    evaluate_model("Decision Tree Regressor", y_test, y_pred_dt),
    evaluate_model("Random Forest Regressor", y_test, y_pred_rf)
]

# Create results table
results_df = pd.DataFrame(results)

# Sort models by RMSE: lower RMSE means better performance
results_df = results_df.sort_values(by="RMSE", ascending=True).reset_index(drop=True)

print("Model Evaluation Results:")
display(results_df)

# Select the best model
best_model = results_df.iloc[0]

print("Best Performing Model:")
print("Model:", best_model["Model"])
print("MAE:", best_model["MAE"])
print("MSE:", best_model["MSE"])
print("RMSE:", best_model["RMSE"])
print("R² Score:", best_model["R² Score"])

# RMSE Comparison
plt.figure(figsize=(8, 5))
plt.bar(results_df["Model"], results_df["RMSE"])
plt.title("Model Comparison Based on RMSE")
plt.xlabel("Models")
plt.ylabel("RMSE")
plt.xticks(rotation=25, ha="right")
plt.show()

# R² Score Comparison
plt.figure(figsize=(8, 5))
plt.bar(results_df["Model"], results_df["R² Score"])
plt.title("Model Comparison Based on R² Score")
plt.xlabel("Models")
plt.ylabel("R² Score")
plt.xticks(rotation=25, ha="right")
plt.show()

# Actual vs Predicted for the best model
best_model_name = best_model["Model"]

if best_model_name == "Mean Baseline":
    best_predictions = baseline_pred
elif best_model_name == "Linear Regression":
    best_predictions = y_pred
elif best_model_name == "Decision Tree Regressor":
    best_predictions = y_pred_dt
else:
    best_predictions = y_pred_rf

plt.figure(figsize=(6, 6))
plt.scatter(y_test, best_predictions, alpha=0.6)
plt.xlabel("Actual Hotel Prices")
plt.ylabel("Predicted Hotel Prices")
plt.title("Actual vs Predicted Prices - " + best_model_name)
plt.show()

NameError: name 'y_train' is not defined